In [3]:
import sys
import os
sys.path.append(os.path.abspath('../'))

In [2]:
import pandas as pd
import plotly.express as px
df = pd.read_csv('..\\data\\raw\\processed\\tickets_dataset.csv')

df.drop(columns=['tag_9'], inplace=True)
df.loc[df["business_type"].isin(["Pit Services", "Adobe Photoshop 2024", "_IT_Services_", "IT Consulting Service"]), "business_type"] = "IT Services"

# Connecting tags into a single column and creating a new column for the combined context of the problem
tags = [col for col in df.columns if col.startswith('tag_')]
df["tags"] = df[tags].apply(lambda x: x.dropna().tolist(), axis=1)

# Combining 'subject' and 'body' into a single column 'context_problem'
df["context_problem"] = df["subject"].fillna("") + " " + df["body"].fillna("")
df_with_tags = df.drop(columns=tags + ['subject', 'body'])

# Reordering columns to have 'context_problem' first
new_order = ['context_problem'] + df_with_tags.columns[:-1].to_list()
df_with_tags = df_with_tags[new_order]

In [5]:
from src.data_loader import DataLoader
loader = DataLoader('..\\data\\raw\\processed\\tickets_dataset.csv')
df = loader.get_clean_ticket_dataset()
df

INFO:src.data_loader:Data loaded successfully from ..\data\raw\processed\tickets_dataset.csv
INFO:src.data_loader:Data cleaned successfully


,context_problem,answer,type,queue,priority,language,business_type,tags
0,Problema crítico del servidor requiere atenció...,Estamos investigando urgentemente el problema ...,Incident,Technical Support,high,es,IT Services,"[Urgent Issue, Service Disruption, Incident Re..."
1,Anfrage zur Verfügbarkeit des Dell XPS 13 9310...,"Sehr geehrter <name>,\n\nvielen Dank, dass Sie...",Request,Customer Service,low,de,Tech Online Store,"[Sales Inquiry, Product Support, Customer Serv..."
2,Erro na Autocompletação de Código do IntelliJ ...,"Prezado <name>,\n\nObrigado por entrar em cont...",Incident,Technical Support,high,pt,IT Services,"[Technical Support, Software Bug, Problem Reso..."
3,Urgent Assistance Required: AWS Service Dear I...,"Dear <name>,\n\nThank you for reaching out reg...",Request,IT Support,high,en,IT Services,"[IT Support, Urgent Issue, Service Notificatio..."
4,Problème d'affichage de MacBook Air Cher équip...,"Cher <name>,\n\nMerci de nous avoir contactés ...",Incident,Product Support,low,fr,Tech Online Store,"[Technical Support, Product Support, Hardware ..."
...,...,...,...,...,...,...,...,...
3995,Problem mit der HP DeskJet 3755 WLAN-Verbindun...,"Sehr geehrter <name>, bitte stellen Sie sicher...",Problem,Customer Service,low,de,Tech Online Store,"[Technical Support, Product Support, Hardware ..."
3996,Problemas com a impressora HP DeskJet 3755 Olá...,Assunto: Re: Problemas com a impressora HP Des...,Incident,Product Support,medium,pt,Tech Online Store,"[Technical Support, Printer Issues, Hardware F..."
3997,Problema urgente con el envío Estimado equipo ...,"Estimado <name>,\n\nNos disculpamos por el ret...",Incident,Service Outages and Maintenance,high,es,Online Store,"[Shipping Delay, Customer Service, Order Issue..."
3998,"Cher Service Client, nous rencontrons des pro...","Cher <name>,\n\nMerci de nous avoir contactés ...",Incident,Technical Support,high,fr,IT Services,"[Technical Support, IT Support, Network Issue,..."


# EDA

In [2]:
df_with_tags.head()

,context_problem,answer,type,queue,priority,language,business_type,tags
0,Problema crítico del servidor requiere atenció...,Estamos investigando urgentemente el problema ...,Incident,Technical Support,high,es,IT Services,"[Urgent Issue, Service Disruption, Incident Re..."
1,Anfrage zur Verfügbarkeit des Dell XPS 13 9310...,"Sehr geehrter <name>,\n\nvielen Dank, dass Sie...",Request,Customer Service,low,de,Tech Online Store,"[Sales Inquiry, Product Support, Customer Serv..."
2,Erro na Autocompletação de Código do IntelliJ ...,"Prezado <name>,\n\nObrigado por entrar em cont...",Incident,Technical Support,high,pt,IT Services,"[Technical Support, Software Bug, Problem Reso..."
3,Urgent Assistance Required: AWS Service Dear I...,"Dear <name>,\n\nThank you for reaching out reg...",Request,IT Support,high,en,IT Services,"[IT Support, Urgent Issue, Service Notificatio..."
4,Problème d'affichage de MacBook Air Cher équip...,"Cher <name>,\n\nMerci de nous avoir contactés ...",Incident,Product Support,low,fr,Tech Online Store,"[Technical Support, Product Support, Hardware ..."


In [24]:
['context_problem'] + df_with_tags.columns[:-1].to_list()

['context_problem',
 'answer',
 'type',
 'queue',
 'priority',
 'language',
 'business_type',
 'tags']

In [12]:
df['priority'].value_counts()

priority
high      1649
medium    1603
low        748
Name: count, dtype: int64

In [7]:
df['language'].value_counts()

language
en    1391
de     848
es     812
fr     476
pt     473
Name: count, dtype: int64

In [16]:
df['tag_9'].isnull().sum()

np.int64(4000)

In [44]:
df['business_type'].value_counts()

business_type
IT Services                     1732
Tech Online Store               1395
IT Consulting Firm               391
Software Development Company     311
Online Store                     171
Name: count, dtype: int64

In [2]:
df['queue'].value_counts()

queue
Technical Support                  1317
Product Support                     690
Customer Service                    627
IT Support                          445
Billing and Payments                338
Returns and Exchanges               197
Service Outages and Maintenance     141
Sales and Pre-Sales                 137
General Inquiry                      55
Human Resources                      53
Name: count, dtype: int64

In [8]:
# String EDA
mean_subject = df['subject'].str.len().mean()
mean_body = df['body'].str.len().mean()
mean_answer = df['answer'].str.len().mean()

median_subject = df['subject'].str.len().median()
median_body = df['body'].str.len().median()
median_answer = df['answer'].str.len().median()

max_subject = df['subject'].str.len().max()
max_body = df['body'].str.len().max()
max_answer = df['answer'].str.len().max()

min_subject = df['subject'].str.len().min()
min_body = df['body'].str.len().min()
min_answer = df['answer'].str.len().min()

print(f"Subject - Mean: {mean_subject}, Median: {median_subject}, Max: {max_subject}, Min: {min_subject}")
print(f"Body - Mean: {mean_body}, Median: {median_body}, Max: {max_body}, Min: {min_body}")
print(f"Answer - Mean: {mean_answer}, Median: {median_answer}, Max: {max_answer}, Min: {min_answer}")

Subject - Mean: 47.640249080101896, Median: 47.0, Max: 320.0, Min: 1.0
Body - Mean: 758.5026256564141, Median: 654.0, Max: 2843.0, Min: 26.0
Answer - Mean: 739.34825, Median: 697.5, Max: 2391, Min: 52


In [27]:
mean_context = df_with_tags['context_problem'].str.len().mean()
median_context = df_with_tags['context_problem'].str.len().median()
max_context = df_with_tags['context_problem'].str.len().max()
min_context = df_with_tags['context_problem'].str.len().min()

print(f"Context Problem - Mean: {mean_context}, Median: {median_context}, Max: {max_context}, Min: {min_context}")

Context Problem - Mean: 801.39125, Median: 699.0, Max: 2923, Min: 17


In [3]:
df.isnull().sum()

subject           467
body                1
answer              0
type                0
queue               0
priority            0
language            0
business_type       0
tag_1               0
tag_2               0
tag_3               0
tag_4               1
tag_5             637
tag_6            1819
tag_7            2955
tag_8            3731
dtype: int64

In [26]:
df_with_tags.isnull().sum()

context_problem    0
answer             0
type               0
queue              0
priority           0
language           0
business_type      0
tags               0
dtype: int64

In [37]:
df_with_tags.loc[df_with_tags['context_problem'].str.len() < 50]

,context_problem,answer,type,queue,priority,language,business_type,tags
194,Verbessern Sie die Effizienz des Ticket-Systems.,Vielen Dank für Ihren Vorschlag. Wir arbeiten ...,Change,Product Support,high,de,Software Development Company,"[Customer Feedback, Feature Request, General I..."
377,Bitte reduzieren Sie die Störung des Dienstes.,Vielen Dank für Ihr Feedback. Wir werden daran...,Change,Service Outages and Maintenance,high,de,IT Services,"[Service Disruption, Service Recovery, Custome..."
1637,Google Workspace,"Of course, how can I help you with Google Work...",Request,Product Support,high,en,Software Development Company,"[Technical Support, Product Support, General I..."
1743,Please reduce service disruptiveness.,Thank you for your feedback. We'll work on min...,Change,Service Outages and Maintenance,high,en,IT Services,"[Service Disruption, Customer Feedback, Proble..."
2120,Issue with the battery on a MacBook Air M1.,We appreciate your reaching out. Could you ple...,Request,Technical Support,high,en,Online Store,"[Technical Support, Product Support, Hardware ..."
2416,O software fica sem resposta ao exportar.,Certifique-se de que o software está atualizad...,Incident,Product Support,high,pt,Tech Online Store,"[Software Bug, Technical Support, Problem Reso..."
2751,Dificultad con la batería del MacBook Air M1.,Agradecemos que nos haya escrito. Le solicitam...,Request,Technical Support,high,es,Online Store,"[Technical Support, Product Support, Hardware ..."
2908,Melhorar a eficiência do sistema de tíquetes.,Obrigado pela sua sugestão. Estamos trabalhand...,Change,Product Support,high,pt,Software Development Company,"[General Inquiry, Feature Request, Customer Fe..."
2929,Client can't log in to Jira Software 8.20.,We're looking into your login issue. Please tr...,Request,Technical Support,high,en,Software Development Company,"[Login Issue, Technical Support, Problem Resol..."
3857,Problem with the battery on your MacBook Air M1.,We appreciate your reaching out to us. Could y...,Request,Technical Support,high,en,Online Store,"[Technical Support, Hardware Failure, Product ..."


In [31]:
df_with_tags.loc[df_with_tags['priority'] == "medium"].head()

,context_problem,answer,type,queue,priority,language,business_type,tags
8,Surface Pro 7 Issue Dear Tech Online Store Sup...,"Dear <name>,\n\nThank you for reaching out to ...",Incident,Product Support,medium,en,Tech Online Store,"[Technical Support, Product Support, Software ..."
9,Problèmes de déploiement avec les ressources A...,"Cher <name>,\n\nMerci de nous avoir contactés ...",Problem,IT Support,medium,fr,IT Services,"[Technical Support, Problem Resolution, Servic..."
10,Request for software development consultation ...,"Dear <name>,\n\nThank you for reaching out reg...",Request,Technical Support,medium,en,IT Services,"[IT Support, Technical Guidance, Problem Resol..."
12,Urgent AWS Deployment Issues Dear IT Services ...,"Dear <name>,\n\nThank you for contacting IT Se...",Request,Technical Support,medium,en,IT Services,"[IT Support, Service Disruption, Urgent Issue,..."
14,Probleme mit der drahtlosen Funktion des Epson...,"Lieber <name>,\n\nvielen Dank für Ihre Nachric...",Incident,Product Support,medium,de,Tech Online Store,"[Technical Support, Product Support, Software ..."


In [45]:
px.histogram(df_with_tags, x='priority', title='Distribution of Ticket Priorities')

# Analysis of data